# Voz de Asuka para Athena (Piper, entrenada en Kaggle gratis)

1. **Settings → Accelerator → GPU T4** e **Internet → On** (móvil verificado en Kaggle).
2. **Add Input**: el dataset `voz-asuka-datos` (el zip que dejó Athena en Descargas).
3. **Save Version → Save & Run All (Commit)**. Tarda unas 6 horas; puedes cerrar el navegador.
4. Al terminar, en **Output** descarga `es_ES-asuka-medium.onnx` y `es_ES-asuka-medium.onnx.json` y avisa a Claude, que los pasa al portátil.

Cada paso comprueba que ha ido bien y, si no, se para con el error a la vista (no sigue a ciegas).

In [ ]:
import subprocess, sys
def sh(cmd, ver=40):
    """Ejecuta y, si falla, enseña las últimas líneas y PARA el cuaderno."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    salida = (r.stdout + r.stderr).strip().splitlines()
    if r.returncode != 0:
        print("\n".join(salida[-ver:]))
        raise RuntimeError(f"Ha fallado: {cmd[:120]}")
    return salida

print("\n".join(sh("nvidia-smi --query-gpu=name,memory.total --format=csv")))
sh("apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build")
sh("rm -rf /kaggle/working/piper1-gpl && git clone -q https://github.com/OHF-voice/piper1-gpl.git /kaggle/working/piper1-gpl")
%cd /kaggle/working/piper1-gpl
sh("pip install -q -e '.[train]' 'scikit-build<1' 'cmake>=3.18,<4' 'ninja>=1,<2'")
sh("./build_monotonic_align.sh")
sh("python3 setup.py build_ext --inplace")
# arreglos para esta versión de Piper: sin el guardado por «val_mos» (rompe con Lightning 2.6) y exportación clásica a ONNX
import re
p = "src/piper/train/__main__.py"; s = open(p).read()
open(p, "w").write(re.sub(r'\n\s*ModelCheckpoint\(\s*monitor="val_mos".*?\),\n', "\n", s, count=1, flags=re.S))
p = "src/piper/train/export_onnx.py"; s = open(p).read()
if "dynamo=False" not in s:
    open(p, "w").write(s.replace("opset_version=OPSET_VERSION,", "opset_version=OPSET_VERSION,\n        dynamo=False,"))
sh("python3 -c 'from piper import espeakbridge; from piper.train.vits.monotonic_align import maximum_path; print(1)'")
print("instalación correcta")

In [ ]:
import glob, os
from huggingface_hub import hf_hub_download
csv = glob.glob('/kaggle/input/**/metadata.csv', recursive=True)[0]
audio = os.path.join(os.path.dirname(csv), 'wavs')
print(csv, len(open(csv).read().splitlines()), 'frases;', len(os.listdir(audio)), 'audios')
ck = hf_hub_download('rhasspy/piper-checkpoints', 'es/es_ES/davefx/medium/epoch=5629-step=1605020.ckpt',
                     repo_type='dataset', local_dir='/kaggle/working/base')
os.makedirs('/kaggle/working/asuka', exist_ok=True)
print('voz de partida:', ck)

In [ ]:
# ~0,5 h de audio: 1.200 épocas desde una voz española ya entrenada (unas 6 h en una T4)
comun = (f"python3 -m piper.train fit --data.voice_name asuka --data.csv_path '{csv}' --data.audio_dir '{audio}' "
         "--model.sample_rate 22050 --data.espeak_voice es --data.cache_dir /kaggle/working/cache "
         "--data.config_path /kaggle/working/asuka/config.json --data.batch_size 24 --model.mos_metric none "
         "--trainer.max_epochs 1200 --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 32 "
         "--trainer.default_root_dir /kaggle/working/entreno")
try:
    sh(comun + f" --model.warmstart_ckpt '{ck}'", ver=60)
except RuntimeError:
    print("\nNo ha podido partir de la voz española; sigo desde cero con más épocas.")
    sh(comun.replace("--trainer.max_epochs 1200", "--trainer.max_epochs 2500"), ver=60)
print("entrenamiento terminado")

In [ ]:
import glob, os, shutil
ult = (glob.glob('/kaggle/working/entreno/**/last.ckpt', recursive=True)
       or sorted(glob.glob('/kaggle/working/entreno/**/*.ckpt', recursive=True), key=os.path.getmtime))[-1]
print('checkpoint:', ult)
sh(f"python3 -m piper.train.export_onnx --checkpoint '{ult}' --output-file /kaggle/working/es_ES-asuka-medium.onnx")
shutil.copy('/kaggle/working/asuka/config.json', '/kaggle/working/es_ES-asuka-medium.onnx.json')
for d in ('/kaggle/working/cache', '/kaggle/working/base'):          # que en Output solo quede lo útil
    shutil.rmtree(d, ignore_errors=True)
print("\n".join(sh("ls -la /kaggle/working/*.onnx*")))

In [ ]:
# Prueba: escucha cómo suena
from piper import PiperVoice
import wave, IPython
v = PiperVoice.load('/kaggle/working/es_ES-asuka-medium.onnx')
with wave.open('/kaggle/working/prueba.wav', 'wb') as w:
    v.synthesize_wav('Hola, soy Athena. Desde hoy hablo con la voz de Asuka. ¿Qué te parece?', w)
IPython.display.Audio('/kaggle/working/prueba.wav')